In [1]:
import pandas as pd
import numpy as np
import sys
import glob
from scipy.ndimage import median_filter
from scipy import signal
import math
import matplotlib.pyplot as plt

In [2]:
sys.path.append("/Users/pnr5sh/Documents/phd/mmmp/")
import sidchaini.sidhelpers as sidhelpers

In [3]:
#reading in meta data from my dir
dataset = pd.read_csv('multipeak_dataset_metadata.csv', header='infer')
dataset['ztf_name'] = ['ZTF']*len(dataset)
dataset.columns

Index(['wise_objid', 'IAU name', 'Internal name/s', 'Obj. RA', 'Obj. DEC',
       'Obj. Type', 'Redshift', 'Spec. ID', 'Obs-date', 'JD', 'Phase (days)',
       'From', 'Telescope', 'Instrument', 'Observer/s', 'Reducer/s',
       'Source group', 'Public', 'Associated groups', 'End prop. period',
       'Ascii file', 'Fits file', 'Spec. type', 'Spec. quality',
       'Extinction-Corrected', 'WL Medium', 'WL Units',
       'Flux Unit Coefficient', 'Spec. units', 'Flux Calibrated By',
       'Exp-time', 'Aperture (slit)', 'HA', 'Airmass', 'Dichroic', 'Grism',
       'Grating', 'Blaze', 'Lambda-min', 'Lambda-max', 'Del-Lambda', 'Contrib',
       'Publish', 'Remarks', 'Created by', 'Creation date', 'mjd', 'peak_mjd',
       'peak_mag', 'peak_filt', 'double-peaked', 'ztf_name'],
      dtype='object')

In [4]:
#manually renaming objects w/o pre-loaded ZTF names
dataset.loc[dataset['IAU name']=='SN 2024zsw', 'Internal name/s'] = 'ZTF24abpdzvm'
dataset.loc[dataset['IAU name']=='SN 2020urc', 'Internal name/s'] = 'ZTF20acgiglu'
# dataset.loc[dataset['IAU name']=='SN 2018cew', 'Internal name/s'] = 'ZTF...' #doesnt have name?
dataset.loc[dataset['IAU name']=='SN 2024abmk', 'Internal name/s'] = 'ZTF24absznoi'
dataset.loc[dataset['IAU name']=='SN 2024abtu', 'Internal name/s'] = 'ZTF24abtnkbi'
dataset.loc[dataset['IAU name']=='SN 2024zzy', 'Internal name/s'] = 'ZTF24abqqven'

In [5]:
#populating the ztf_name feature of the dataset by extracting from internal name/s feature

intnamelist = dataset['Internal name/s'].to_list()
iaunames = dataset['IAU name'].to_list()

name_list = [(str(intnamelist[i]), str(iaunames[i])) for i in range(len(intnamelist))]

for pair in name_list:
    intname = pair[0]
    iauname = pair[1]
    varX = "ZTF"
    index = intname.find(varX)
    ztf_name = intname[index:index+12]
    if ztf_name[0] != 'Z':
        print(f'warning: could not parse out ZTF name from internal names list for {iauname}')
    else:
        dataset.loc[dataset['IAU name']==iauname, 'ztf_name'] = ztf_name

In [60]:
#creating our version of the ZTF BTS metadata info file
# ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,peakabs,duration,rise,fade,type,redshift,b,A_V
# already have: ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,type,redshift, double-peaked
# want to get: A_V, peakabs
# ignoring: duration, rise, fade, b

#TODO: calc extinction
#TODO: calc abs mag (hubble flow?)

bs_ZTFBTS = pd.DataFrame(columns=['ZTFID','IAUID','RA','Dec','peakt','peakfilt',
                                  'peakmag','type','redshift','double-peaked'])

znames, iaunames, ras, decs, pmjds, pfilts, pmags, types, zs, dps = [],[],[],[],[],[],[],[],[],[]
for i,obj in enumerate(dataset['ztf_name'].unique()):
    if obj=='ZTF':
        print(f'No ZTF name, skipping {dataset.loc[dataset['ztf_name']=='ZTF', 'IAU name'].iloc[0]} for now...')
    else:
        data = dataset.loc[dataset['ztf_name']==obj]
        
        #some objs have repeat entries, just taking the first one
        znames.append(data['ztf_name'].iloc[0])
        iaunames.append(data['IAU name'].iloc[0][0:2]+data['IAU name'].iloc[0][3:])
        ras.append(data['Obj. RA'].iloc[0])
        decs.append(data['Obj. DEC'].iloc[0])
        pmjds.append(data['peak_mjd'].iloc[0])
        pfilts.append(data['peak_filt'].iloc[0][-1])
        pmags.append(data['peak_mag'].iloc[0])
        types.append(data['Obj. Type'].iloc[0])
        zs.append(data['Redshift'].iloc[0])
        dps.append(data['double-peaked'].iloc[0])
        

bs_ZTFBTS['ZTFID'] = znames
bs_ZTFBTS['IAUID'] = iaunames
bs_ZTFBTS['RA'] = ras
bs_ZTFBTS['Dec'] = decs
bs_ZTFBTS['peakt'] = pmjds
bs_ZTFBTS['peakfilt'] = pfilts
bs_ZTFBTS['peakmag'] = pmags
bs_ZTFBTS['type'] = types
bs_ZTFBTS['redshift'] = zs
bs_ZTFBTS['double-peaked'] = dps

No ZTF name, skipping SN 2018cew for now...


In [61]:
bs_ZTFBTS

,ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,type,redshift,double-peaked
0,ZTF23ableqsp,AT2023vth,269.143338,8.043701,60237.389119,g,17.544216,FBOT,0.074700,0
1,ZTF18aahhzqn,SN2018avy,178.754792,32.075350,58225.156007,r,18.147050,SN Ib/c,0.031000,1
2,ZTF18aakuewf,SN2018bcc,243.594375,35.917889,58231.129356,g,17.692159,SN Ibn,0.063600,0
3,ZTF18abojpnr,SN2018fzn,297.487120,59.592827,58370.051276,g,19.171934,SN IIb,0.037500,0
4,ZTF18absliyc,SN2018gcj,6.005904,-1.223377,58371.685414,g,18.892202,SN Ic,0.080000,0
...,...,...,...,...,...,...,...,...,...,...
97,ZTF24abzgggz,SN2024afbf,170.447324,46.128450,60675.269521,g,19.457773,SN Ic,0.030000,0
98,ZTF24aaymkrs,SN2024rbc,3.089359,31.063392,60537.304655,g,17.396463,SN Ib,0.016000,0
99,ZTF24abpdzvm,SN2024zsw,20.793617,19.570172,60632.542341,g,19.196814,SN IIb,0.032286,1
100,ZTF24abqqven,SN2024zzy,150.722002,8.437942,60624.468310,r,17.892795,SN Ib,0.030487,0


In [ ]:
#maven needs light curves files w/ 4 columns: time,mag,magerr,band
#   where time is in mjd and filter is either g, R

# dropping bad obs
# centering LCs around date of first spectra (-50, +150 days)
# creating mag and mag_err cols
# dropping points w/ e_mag>2 mags
# creating mjd col

def create_ztf_lc_dfs(datadir, sn_type, check_names=False, save_df=True):
        files = sorted(glob.glob(datadir+'*_fp_lc.txt'))
        ztf_info_df = pd.read_csv(f'ztf_info_files/ztf_fp_info_for_{sn_type}.csv')

        if (sn_type == 'SN IIb') or (sn_type =='SN IIn') or (sn_type == 'SN Ibc'):
                startindx = 18
        elif (sn_type == 'SN Ic') or (sn_type =='SN Ib'):
                startindx = 17
        elif (sn_type == 'SLSN-I') or (sn_type == 'SN IbnIcn') or (sn_type == 'SN Ic-pec'):
                startindx = 20
        elif (sn_type == 'SLSN-II') or (sn_type =='SN Ibc Ca-rich'):
                startindx = 21
        elif sn_type == 'FBOT':
                startindx = 19
        else:
                print(f'SN type {sn_type} unsupported or not in format SN XX / SLSN-XX')
                return

        sn_names, lc_dfs = [],[]
        for file in files:
                sn_name = file[startindx:-10] #NOTE: FIRST INDEX CHANGES W/ DATADIR NAME LENGTH
                if check_names:
                        print(sn_name)
                        continue
                
                sn_names.append(sn_name)

                cols = ['index', 'field', 'ccdid', 'qid', 'filter', 'pid', 'infobitssci', 'sciinpseeing', 'scibckgnd', 'scisigpix',
                        'zpmaginpsci', 'zpmaginpsciunc', 'zpmaginpscirms', 'clrcoeff', 'clrcoeffunc', 'ncalmatches', 'exptime',
                        'adpctdif1', 'adpctdif2', 'diffmaglim', 'zpdiff', 'programid', 'jd', 'rfid', 'forcediffimflux', 'forcediffimfluxunc',
                        'forcediffimsnr', 'forcediffimchisq', 'forcediffimfluxap', 'forcediffimfluxuncap', 'forcediffimsnrap', 'aperturecorr',
                        'dnearestrefsrc', 'nearestrefmag', 'nearestrefmagunc', 'nearestrefchi', 'nearestrefsharp', 'refjdstart', 'refjdend', 'procstatus']
                df = pd.read_csv(file, names=cols, header=None, sep=" ", skiprows=54)
                df = df.set_index(df['index'])  # manually setting indeces
                df = df.drop(columns=['index']) # drop duplicated index 
                df = df[(df['infobitssci'] < 33554432) & (df['scisigpix'] <= 25) & (df['sciinpseeing'] <= 4) & (df['forcediffimflux']!=-99999.0)].reset_index(drop=True) #clean according to docs

                # cut df down to -50 days to +365 days centered on date of first spectra
                # time window included in ztf_fp_info*csv files for each obj type
                obj = ztf_info_df.loc[ztf_info_df['obj_name'].str[3:]==sn_name]
                start_jd = obj['jd_start'].iloc[0]
                end_jd = obj['jd_end'].iloc[0]
                df_cut = df.loc[(df['jd']<end_jd)&(df['jd']>start_jd)] #only selecting points that fall within specified time window
                df_cut = df_cut.reset_index(drop=True)
                df_cut = df_cut.infer_objects() #infering dtype of columns

                mag = df_cut['zpdiff'] - 2.5 * np.log10(df_cut['forcediffimflux'])
                sigma_mag = 1.0857 * df_cut['forcediffimfluxunc']/df_cut['forcediffimflux']
                df_cut['mag'] = mag
                df_cut['e_mag'] = sigma_mag
                df_cut = df_cut.loc[(df_cut['forcediffimflux']>0) & (df_cut['e_mag']<2)].reset_index(drop=True) #only selecting points w/ non-nan mags and errorbars less than 2 mags
                df_cut['mjd'] = df_cut['jd']-2400000.5

                # saving LC in maven-friendly format w/ ZTF name as file name
                # if obj has no ZTF internal name, saved w/ IAU name and will need to look up personally 
                #       and update the metadata
                if save_df:
                        if 'SN '+sn_name in dataset['IAU name'].to_list(): #only converting objs in our "good" sample
                                smol_df = df_cut[['mjd', 'mag', 'e_mag', 'filter']]
                                smol_df = smol_df.rename(columns={"mjd": "time", "e_mag": "magerr", "filter":"band"})
                                smol_df.loc[smol_df['band']=='ZTF_g', 'band'] = 'g'
                                smol_df.loc[smol_df['band']=='ZTF_r', 'band'] = 'R'

                                ztf_name = dataset.loc[dataset['IAU name']=='SN '+sn_name, 'ztf_name'].iloc[0]
                                if len(ztf_name)<4:
                                        print(f'WARNING: SN {sn_name} has no ZTF name; need for maven')
                                        smol_df.to_csv(f'maven_data/lightcurves/SN{sn_name}.csv',index=False)
                                else:
                                        smol_df.to_csv(f'maven_data/lightcurves/{ztf_name}.csv',index=False)                                                             

                lc_dfs.append(df_cut)

        return sn_names, lc_dfs

In [15]:
# sn_names_ib, lc_dfs_ib = create_ztf_lc_dfs('./ztf_fp_data/ib/','SN Ib')
# sn_names_ic, lc_dfs_ic = create_ztf_lc_dfs('./ztf_fp_data/ic/','SN Ic')
sn_names_iib, lc_dfs_iib = create_ztf_lc_dfs('./ztf_fp_data/iib/','SN IIb', check_names=False, save_df=True)
# sn_names_carich, lc_dfs_carich = create_ztf_lc_dfs('./ztf_fp_data/carich/','SN Ibc Ca-rich', check_names=False)
# sn_names_ibc, lc_dfs_ibc = create_ztf_lc_dfs('./ztf_fp_data/ibc/','SN Ibc', check_names=False)
# sn_names_ibncn, lc_dfs_ibncn = create_ztf_lc_dfs('./ztf_fp_data/ibncn/','SN IbnIcn', check_names=False)
# sn_names_icpec, lc_dfs_icpec = create_ztf_lc_dfs('./ztf_fp_data/icpec/','SN Ic-pec', check_names=False)

/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
#NOTE: some objs have more than 1 spectra. Need to figure out how wanna treat that
#NOTE: spectra all have different naming conventions, need to figure out how to read in and assoc. w/ obj in dataset metadata

header = ['wavelength', 'flux', 'fluxerr']
#spectra files are either names for ZTF name or IAU name
spectrafile = pd.read_csv('./wiserep_spectra/iib_spectra/spectra/ZTF18abojpnr_2458351.897_P60_SEDM_ZTF.ascii', 
                          delimiter=' ', skiprows=168, header=0, names=header)
spectrafile['smooth_flux'] = [0]*len(spectrafile)

no_spec = len(dataset.loc[dataset['ztf_name']=='ZTF18abojpnr', 'Del-Lambda']) #Del-Lambda is wavelength spacing

#values taken from DASH paper
w0 = 3500
w1 = 10000
nw = 1024
w_den = (w1 - w0)/nw 
smooth = 6
dwlog = np.log(w1/w0)/nw #0.00102
a = nw / np.log(w1/w0) #975.40
b = -nw*np.log(w0)/np.log(w1/w0) #-7959.796

binned_waves = []
for i in range(no_spec):
    #move the spectra to have mean = 0
    mean = np.mean(spectrafile['flux'])
    flux_cen0 = spectrafile['flux']-mean

    # set values in all columns, with wavelength outside of 3500-10000, to 0
    spectrafile.loc[(spectrafile['wavelength']<3500)|(spectrafile['wavelength']>10000), ['']] = 0

    #apply low-pass median filter to smooth spectrum
    del_lam = dataset.loc[dataset['ztf_name']=='ZTF18abojpnr', 'Del-Lambda'].iloc[i]
    window_size = math.ceil( (w_den / del_lam) * smooth ) #rounds value to next highest integer
    smooth_spec = spectrafile['flux'].rolling(window_size, center=True).median()
    spectrafile['smooth_flux'] = smooth_spec
    
    ##### ATTEMPTING TO CREATE LOG WAVELENGTH BINS TO THEN BIN SPECTRA, 
    ##### BUT RESULTS UNPHYSICAL
    #log-spacing wavelength
    log_wave = np.logspace(np.log(3500), np.log(10000), num=1024, endpoint=True, base=np.e)
    binned_fluxes = []
    for j, rightbin in enumerate(log_wave[1:]):
        leftbin = log_wave[j]
        binned_flux = spectrafile.loc[(leftbin<spectrafile['wavelength'])&
                                      (spectrafile['wavelength']<rightbin),
                                      'smooth_flux'].sum()
        binned_fluxes.append(binned_flux)
    
    #NOTE : PICK UP HERE
    %matplotlib qt
    temp_flux = 
    plt.plot(log_wave[1:], binned_fluxes)

    # for j in range(nw+1): 
    #     wlogn = w0*np.log(j*dwlog)
    #     binned_wave = a*np.log(wlogn)+b
        # print(j, wlogn, np.log(wlogn), a*np.log(wlogn), 3500+binned_wave)
        # binned_waves.append(binned_wave)

    # fitting and dividing out 13-point cubic spline fit to remove continuum
